PINNs - Redes Neurais Informadas por Física
====================

# Teoria 

## O que são?

Uma __Physics-Informed Neural Network__ (_PINN_) ou, em português claro, _Rede Neural Informada por Física_ é um framework de aprendizado profundo em que uma rede neural é treinada para aproximar a solução de uma equação diferencial ao inserir leis físicas diretamente na função de perda. 

A proposta primária é justamente incorporar as limitações físicas fundamentais, normalmente na forma de equações diferenciais parciais não lineares (EDPs não lineares), ao invés de basear-se apenas em um grande volume de dados para guiar o processo de aprendizagem. 

## Breve Histórico

### Antecedentes

A ideia de utilizar redes neurais para resolver equações diferenciais antecede o acrônimo _PINN_ por praticamente duas décadas. O trabalho considerado fundamental é de Lagaris, Likas e Fotiadis (1998) [1], que propuseram escrever _trial solutions_ como somas de duas partes - uma satisfazendo as condições de limite exatamente e uma segunda envolvendo uma rede de feedfoward (aquelas em que os dados fluem em uma única direção sem ciclos ou feedback). 

 - Para essa aplicação, é interessante entender o que são as _trial solutions_. A tradução mais comum é **função tentativa**, uma função construída artificialmente para garantir que ela satisfaça automaticamente as condições de contorno e que dependa de uma função ajustável a ser otimizada.

Eles demonstraram essa abordagem em EDOs e EDPs, comparando-se favoravelmente contra outros métodos computacionais, apesar dela ser consideravelmente anterior à era moderna do aprendizado profundo. 

### A introdução do framework

Os responsáveis pela construção moderna das _PINNs_ foram Maziar Raissi, Paris Perdikaris e George Kaniadakis. Em artigo publicado em 2019 no *Journal of Computational Physics*[2], a referência acadêmica canômica sobre a matéria, eles fizeram uma contribuição chave; transformaram o problema em um framework de aprendizado multi-tarefa. Agora, a rede neural deveria simultaneamente conferir o ajuste dos dados observáveis **e** minimzar os resíduos das EDPs que modelam o sistema, avaliando as funções em pontos pelo domínio.

O grande salto foi fazer uso da diferenciação automática, presente e disponível em bibliotecas de aprendizado profundo como _TensorFlow_ ou _PyTorch_, para computar de forma mais direta os resíduos das EDPs. Essa inovação forneceu uma vantagem prática com relação aos algoritmos anteriores que seguiam o estilo de Largaris. Além disso, algo que distinguia ainda mais as _PINNs_ desses algoritmos era sua capacidade de não apenas resolver problemas diretamente, masi inversamente e através de descoberta guiada por dados (_data-drive discovery_) das equações que governam os sistemas estudados. 

## Conceitos Centrais

### A função de perda

Após entendermos um pouco a respeito das definições e da história das _PINNs_, devemos esmiuçar os seus meandros para uma compreensão completa. Como foi supracitado, há uma diferença fundamental que separa essas redes de redes neurais tradicionais: a incorporação da física. 

Mas como isso é de fato incorporado do ponto de vista matemático? O que é esse objeto novo que difere tanto as _PINNs_ de outras redes?

Uma rede neural tradiiconal tem como objetivo minimizar a diferença do resultado previsto pela rede e o resultado real (a **perda dos dados**). Estrutura-se uma **função de perda** e o objetivo da rede é, tradicionalmente utilizando os algoritmos de retropropagação e descida do gradiente, minimizá-la. Já em uma _PINN_ devemos introduzir um segundo componente para o objetivo de treinamento: a **perda física** ou a _perda residual_. Dessa forma, reestrutura-se a função de perda; a rede é treinada para minimizar uma função de perda de múltiplos objetivos:

$L_{total} = w_{data}L_{data} \; + \; w_{fisico}L_{fisico} $

Onde:

- $L_{data} \;$  assegura que o modelo se adeque às medidas conhecidas (as condições de contorno e as iniciais);
- $L_{fisico} \;$  penaliza o modelo se as predições violarem as equações físicas que regem o sistema. 

Os $w$'s são os pesos de cada uma das perdas. 

### Resíduos na perda física

#### O que é o resíduo?

Precisamos entender, porém, como é construído $L_{fisico}$ para, então, pensar em métodos de otimização que façam sentido para _PINNs_. Vamos começar por entender o que são **resíduos** no contexto das EDPs. Consideremos uma EDP geral definida pelo operador não-linear $\mathcal{F}$:

$$
\mathcal{F}[u(x,t)] = f (x,t)
$$

Onde $u(x,t)$ é a solução exata para a EDP. Se houve uma solução aproximada, denotada de $\hat{u}(x,t)$, o **resíduo** $R(x,t)$ é definido como a diferença entre o operador aplicado à aproximação e o termo original $f$:

$$
R(x,t) = \mathcal{F}[\hat{u}(x,t)] - f(x,t)
$$

Note que se $\hat{u}$ for a solução exata, o resíduo torna-se 0 em todos os pontos. Em métodos numéricos como o _PINNs_, a ideia é minimazar o resíduo através do domínio. 
Além desse mecanismo clássico, _PINNs_ podem utilizar o resíduo para descoberta de parâmetros, como uma constante física desconhecida $\lambda$. Dessa forma, modifca-se sua fórmula para que $\lambda$ seja incorporado:

$$
R(x, t; \lambda) = \mathcal{F}[\hat{u}(x,t);\lambda] - f(x,t)
$$

É importante fazer uma diferenciação conceitual também. **O resíduo é diferente do erro**. Em uma tarefa comum de aprendizado supervisionado, mede-se o erro por comparação entre a predição do modelo e o dado real. Nesse framework, se a solução exata é desconhecida, não conseguimos calcular o erro e obter métricas para melhorar o modelo. _PINNs_ **não operam com erro, mas sim com resíduo**. O princípio por trás é a **consistência** ao invés da comparação: o resíduo mede o quão bem uma aproximação satisfaz uma lógica interna das equações diferenciais, independentemente se uma solução fechada para o problema existe. 

#### Como ele aparece em _PINNs_?

Em _PINNs_ a própria rede funciona como a função aproximada $\hat{u}(x,t; \theta)$ onde $\theta$ representa os parâmetros de pesos e vieses. O resído é utilizado então para construir o termo de perda física, que informa a rede sobre as limitações físicas do sistema.

Olhemos para o mecanismo passo a passo:

- Pontos de colocação: uma amostra contendo um conjunto de pontos é selecionada no domínio. Esses pontos são chamados de pontos de colocação porque não precisamos dos valores verdadeiros de $u$ para eles, somente como a física se aplica nesses pontos;
- Diferenciação automática (DA): para calcular o resíduo, a rede computa as derivadas com respeito aos dados de entrada. Ao contrário de redes neurais tradicionais que utilizam métodos baseados em grid, que divididem o espaço de dados em um número finito de células para formar uma estrutura, PINNs usam DA para computar as derivadas nos pontos de colocação;
- Constuindo a perda: o resíuo é calculado para todos esses pontos e a função de perda física é, tipicamente, o erro quadrático médio (MSE) dos resíduos:
            $$L_{fisico} = \dfrac{1}{N_f}\sum^{N_f}_{i=1} |R(x_i, t_i)|^{2} $$

- Otimização: o otimizador de escolha ajusta $\theta$ para minimizar a perda física. Quando $L_{fisico} \rightarrow 0$, a rede neural é forçada a satisfazer a EDP.


### Otimização

O conceito que faz com que as _PINNs_ sejam úteis são os métodos de diferenciação automática, como a retropropagação, para computar as derivadas. 

Aqui, porém, temos uma questão mais complicada de **otimização**. Redes neurais comuns quase exclusivamente fazem uso de métodos de primeira ordem, como Adam (_Adaptive Moment Estimation_), SGD (_Stochastic Gradient Descent_) ou Nesterov para realizar a otimização. Eles tem algumas vantagens:

- Rápidos
- Baixo custo de memória
- Escalonáveis para muitos parâmetros

Mas para _PINNs_, uma característica faz com que esses otimizadores não sejam os mais apropriados. O termo físico que inserimos na função de perda possui deirvadas de ordem superior (algo do tipo $\dfrac{\partial ² u}{ \partial x²}$ ou superior), o que faz com que a imagem da função de perda torne-se, por vezes, não bem comportada o suficiente para que usemos otimizadores de primeira ordem. Eles podem ficar presos em alguma condição ou demorar muito para convergir em uma resposta. Por isso, é comum que se utilizem otimizadores de segunda ordem ou, pelo menos, algo próximo. Esses algortimos utilizam ambos gradiante e  Hessiano ($H$), ou a matriz de segundas derivadas. Essa utilização faz com que os algoritmos entendam não só qual a direção do ponto de mínimo, mas como a curvatura está se comportando. 

O método de segunda ordem ideal é o método de Newton, que realiza um salto em direção ao mínimo através da inversão do Hessiano. O impedimento é que, para $N$ parâmetros, o Hessiano é uma matriz $N \times N$. A complexidade de um código que realize essa inversão é cúbica, o que se torna imprático, até para redes de médio porte. É por isso que foi preciso buscar uma alternativa. 

Em _PINNs_, é comum que se utilizem métodos denominados Quasi-Newtonianos, como **BFGS** (Broyden–Fletcher–Goldfarb–Shanno) ou **L-BFGS** (Limited-memory BFGS). Esses métodos aproximam o inverso do Hessiano sem calculá-lo de maneira explícita.  

O método BFGS atualiza uma estimativa do inverso do Hessiano utilizando a **Equação da secante**:
$$B_{k+1} = (I - \rho_k s_k y_k^\top) B_k (I - \rho_k y_k s_k^\top) + \rho_k s_k s_k^\top$$

Onde 

* **$s_k = \theta_{k+1} - \theta_k$**: É a mudança nos parâmetros do modelo (pesos/vieses);
* **$y_k = \nabla L(\theta_{k+1}) - \nabla L(\theta_k)$**: É a mudança no gradiente da função de perda;
* **$\rho_k = \frac{1}{y_k^\top s_k}$**: É um fator escalar de curvatura que garante que a atualização permaneça positiva e definida;
* **$I$**: É a matriz identidade.

A versão L-BFGS limita o uso de memória por não armazenar a matriz $B_k$. Ao invés disso, armazena-se as últimas $m$ versões dos vetores $s_k \text{ e } y_k$. Após isso, utilizam-se dois laços recursivos para computar a direção de ajuste diretamente (LIU, NOCEDAL)[3]. 

De maneira prática, podemos comparar os métodos por convergência. Enquanto métodos lineares (como o Adam) convergem mais rápido, a sua precisão é menor do que um BFGS que converge de forma superlinear. Isso significa que, uma vez perto da solução, ele chega a ela com uma precisão muito maior. 

# Referências

[1]**LAGARIS, I. E.; LIKAS, A.; FOTIADIS, D. I.** *Artificial neural networks for solving ordinary and partial differential equations*. *IEEE Transactions on Neural Networks*, v. 9, n. 5, p. 987–1000, set. 1998. DOI: https://doi.org/10.1109/72.712178.

[2]__RAISSI, M.; PERDIKARIS, P.; KARNIADAKIS, G. E.__ _Physics-informed neural networks: a deep learning framework for solving forward and inverse problems involving nonlinear partial differential equations. Journal of Computational Physics_, v. 378, p. 686–707, 2019. DOI: https://doi.org/10.1016/j.jcp.2018.10.045

[3]**LIU, D. C.; NOCEDAL, J.** On the limited memory BFGS method for large scale optimization. *Mathematical Programming*, v. 45, p. 503–528, 1989. DOI: https://doi.org/10.1007/BF01589116.


